# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- Use Pandas to load, inspect, and clean the dataset appropriately. 
- Transform relevant columns to create measures that address the problem at hand.
- **conduct EDA: visualization and statistical measures to understand the structure of the data**
- **recommend a set of manufacturers to consider as well as specific airplanes conforming to the client's request**
- **discuss the relationship between serious injuries/airplane damage incurred and at least *two* factors at play in the incident. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.**

In [3]:
# loading relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Exploratory Data Analysis  
- Load in the cleaned data

In [ ]:
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

cleaned_path = "data/aviation_accidents_cleaned.csv"
df = pd.read_csv(cleaned_path, parse_dates=["Event.Date"])

print(f"Loaded cleaned dataset: {df.shape}")
display(df.head())

def summarize_groups(frame, group_col, min_n=10):
    summary = (
        frame.groupby(group_col)
        .agg(
            sample_size=(group_col, "size"),
            mean_injury=("Fatal.Serious.Fraction", "mean"),
            mean_destroyed=("Destroyed", "mean"),
            median_passengers=("Total.Passengers", "median"),
        )
        .query("sample_size >= @min_n")
        .sort_values(["mean_injury", "mean_destroyed", "sample_size"], ascending=[True, True, False])
    )
    return summary


## Explore safety metrics across models/makes
- Remember that the client is interested in separate recommendations for smaller airplanes and larger airplanes. Choose a passenger threshold of 20 and separate the plane types. 

In [ ]:
size_summary = (
    df.groupby("Plane.Size")
    .agg(
        accidents=("Plane.Size", "size"),
        mean_injury=("Fatal.Serious.Fraction", "mean"),
        mean_destroyed=("Destroyed", "mean"),
        median_passengers=("Total.Passengers", "median"),
    )
    .sort_index()
)

display(size_summary)

small_df = df[df["Plane.Size"] == "Small"].copy()
large_df = df[df["Plane.Size"] == "Large"].copy()
print("Small-aircraft rows:", len(small_df))
print("Large-aircraft rows:", len(large_df))


#### Analyzing Makes

Explore the human injury risk profile for small and larger Makes:
- choose the 15 makes for each group possessing the lowest mean fatal/seriously injured fraction
- plot the mean fatal/seriously injured fraction for each of these subgroups side-by-side

In [ ]:
small_make_summary = summarize_groups(small_df, "Make.Clean", min_n=10)
large_make_summary = summarize_groups(large_df, "Make.Clean", min_n=10)

display(small_make_summary.head(15))
display(large_make_summary.head(15))

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=False)

sns.barplot(
    data=small_make_summary.head(15).reset_index(),
    x="mean_injury",
    y="Make.Clean",
    ax=axes[0],
    color="steelblue",
)
axes[0].set_title("Small Aircraft: Lowest Mean Fatal/Serious Injury Fraction")
axes[0].set_xlabel("Mean fatal/serious injury fraction")
axes[0].set_ylabel("")

sns.barplot(
    data=large_make_summary.head(15).reset_index(),
    x="mean_injury",
    y="Make.Clean",
    ax=axes[1],
    color="seagreen",
)
axes[1].set_title("Large Aircraft: Lowest Mean Fatal/Serious Injury Fraction")
axes[1].set_xlabel("Mean fatal/serious injury fraction")
axes[1].set_ylabel("")

plt.tight_layout()


**Distribution of injury rates: small makes**

Use a violinplot to look at the distribution of the fraction of passengers serious/fatally injured for small airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
best_small_makes = small_make_summary.head(10).index

plt.figure(figsize=(14, 7))
sns.violinplot(
    data=small_df[small_df["Make.Clean"].isin(best_small_makes)],
    x="Fatal.Serious.Fraction",
    y="Make.Clean",
    order=best_small_makes,
    inner="quartile",
    color="lightsteelblue",
    cut=0,
)
plt.title("Distribution of Injury Fractions for Best-Performing Small-Aircraft Makes")
plt.xlabel("Fatal/serious injury fraction")
plt.ylabel("")
plt.tight_layout()


**Distribution of injury rates: large makes**

Use a stripplot to look at the distribution of the fraction of passengers serious/fatally injured for large airplane makes. Just display makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
best_large_makes = large_make_summary.head(10).index

plt.figure(figsize=(14, 6))
sns.stripplot(
    data=large_df[large_df["Make.Clean"].isin(best_large_makes)],
    x="Fatal.Serious.Fraction",
    y="Make.Clean",
    order=best_large_makes,
    jitter=0.2,
    alpha=0.7,
    color="darkseagreen",
)
plt.title("Large-Aircraft Injury Fractions for Lowest-Risk Makes")
plt.xlabel("Fatal/serious injury fraction")
plt.ylabel("")
plt.tight_layout()


**Evaluate the rate of aircraft destruction for both small and large aircraft by Make.** 

Sort your results and keep the lowest 15.

In [ ]:
small_destroyed_summary = (
    small_make_summary.sort_values(["mean_destroyed", "mean_injury", "sample_size"]).head(15)
)
large_destroyed_summary = (
    large_make_summary.sort_values(["mean_destroyed", "mean_injury", "sample_size"]).head(15)
)

display(small_destroyed_summary)
display(large_destroyed_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 7), sharex=False)

sns.barplot(
    data=small_destroyed_summary.reset_index(),
    x="mean_destroyed",
    y="Make.Clean",
    ax=axes[0],
    color="darkorange",
)
axes[0].set_title("Small Aircraft: Lowest Destroyed Fraction by Make")
axes[0].set_xlabel("Destroyed fraction")
axes[0].set_ylabel("")

sns.barplot(
    data=large_destroyed_summary.reset_index(),
    x="mean_destroyed",
    y="Make.Clean",
    ax=axes[1],
    color="slateblue",
)
axes[1].set_title("Large Aircraft: Lowest Destroyed Fraction by Make")
axes[1].set_xlabel("Destroyed fraction")
axes[1].set_ylabel("")

plt.tight_layout()


#### Provide a short discussion on your findings for your summary statistics and plots:
- Make any recommendations for Makes here based off of the destroyed fraction and fraction fatally/seriously injured
- Comment on the calculated statistics and any corresponding distributions you have visualized.

In [ ]:
def recommendation_table(summary, top_n=5):
    ranked = summary.copy()
    ranked["injury_rank"] = ranked["mean_injury"].rank(method="dense")
    ranked["destroyed_rank"] = ranked["mean_destroyed"].rank(method="dense")
    ranked["combined_rank"] = ranked["injury_rank"] + ranked["destroyed_rank"]
    return ranked.sort_values(["combined_rank", "sample_size"]).head(top_n)

small_recommendations = recommendation_table(small_make_summary, top_n=5)
large_recommendations = recommendation_table(large_make_summary, top_n=5)

print("Recommended small-aircraft makes")
display(small_recommendations[["sample_size", "mean_injury", "mean_destroyed", "combined_rank"]])

print("Recommended large-aircraft makes")
display(large_recommendations[["sample_size", "mean_injury", "mean_destroyed", "combined_rank"]])

print(
    f"Small-aircraft takeaway: {small_recommendations.index[0]} leads the combined ranking, "
    f"with a mean fatal/serious injury fraction of {small_recommendations.iloc[0]['mean_injury']:.3f} "
    f"and destroyed fraction of {small_recommendations.iloc[0]['mean_destroyed']:.3f}."
)
print(
    f"Large-aircraft takeaway: {large_recommendations.index[0]} performs best in the combined ranking, "
    f"with a mean fatal/serious injury fraction of {large_recommendations.iloc[0]['mean_injury']:.3f} "
    f"and destroyed fraction of {large_recommendations.iloc[0]['mean_destroyed']:.3f}."
)


### Analyze plane types
- plot the mean fatal/seriously injured fraction for both small and larger planes 
- also provide a distributional plot of your choice for the fatal/seriously injured fraction by airplane type (stripplot, violin, etc)  
- filter ensuring that you have at least ten individual examples in each model/make to average over

**Larger planes**

In [ ]:
large_model_summary = summarize_groups(large_df, "Plane.Type", min_n=10)

display(large_model_summary.head(15))

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

sns.barplot(
    data=large_model_summary.head(15).reset_index(),
    x="mean_injury",
    y="Plane.Type",
    ax=axes[0],
    color="seagreen",
)
axes[0].set_title("Large Aircraft Types: Lowest Mean Injury Fraction")
axes[0].set_xlabel("Mean fatal/serious injury fraction")
axes[0].set_ylabel("")

sns.stripplot(
    data=large_df[large_df["Plane.Type"].isin(large_model_summary.head(10).index)],
    x="Fatal.Serious.Fraction",
    y="Plane.Type",
    order=large_model_summary.head(10).index,
    ax=axes[1],
    jitter=0.2,
    alpha=0.7,
    color="darkseagreen",
)
axes[1].set_title("Large Aircraft Types: Distribution of Injury Fractions")
axes[1].set_xlabel("Fatal/serious injury fraction")
axes[1].set_ylabel("")

plt.tight_layout()


**Smaller planes**
- for smaller planes, limit your plotted results to the makes with the 10 lowest mean serious/fatal injury fractions

In [ ]:
best_small_makes = small_make_summary.head(10).index
small_model_summary = summarize_groups(
    small_df[small_df["Make.Clean"].isin(best_small_makes)],
    "Plane.Type",
    min_n=10,
)

display(small_model_summary.head(15))

fig, axes = plt.subplots(1, 2, figsize=(16, 8))

sns.barplot(
    data=small_model_summary.head(15).reset_index(),
    x="mean_injury",
    y="Plane.Type",
    ax=axes[0],
    color="steelblue",
)
axes[0].set_title("Small Aircraft Types: Lowest Mean Injury Fraction")
axes[0].set_xlabel("Mean fatal/serious injury fraction")
axes[0].set_ylabel("")

sns.violinplot(
    data=small_df[small_df["Plane.Type"].isin(small_model_summary.head(10).index)],
    x="Fatal.Serious.Fraction",
    y="Plane.Type",
    order=small_model_summary.head(10).index,
    ax=axes[1],
    color="lightsteelblue",
    inner="quartile",
    cut=0,
)
axes[1].set_title("Small Aircraft Types: Distribution of Injury Fractions")
axes[1].set_xlabel("Fatal/serious injury fraction")
axes[1].set_ylabel("")

plt.tight_layout()


### Discussion of Specific Airplane Types
- Discuss what you have found above regarding passenger fraction seriously/ both small and large airplane models.

In [ ]:
print("Best-supported large-aircraft models")
display(large_model_summary.head(10)[["sample_size", "mean_injury", "mean_destroyed"]])

print("Best-supported small-aircraft models")
display(small_model_summary.head(10)[["sample_size", "mean_injury", "mean_destroyed"]])

print(
    f"Among larger aircraft, {large_model_summary.index[0]} has the lowest observed mean fatal/serious injury fraction "
    f"({large_model_summary.iloc[0]['mean_injury']:.3f}) among plane types with at least 10 accidents."
)
print(
    f"Among smaller aircraft, {small_model_summary.index[0]} leads the filtered set with a mean fatal/serious injury fraction "
    f"of {small_model_summary.iloc[0]['mean_injury']:.3f}."
)
print(
    "Because the plots still show spread within each type, the recommendation should favor models that are both low on average "
    "and supported by a reasonable sample size."
)


### Exploring Other Variables
- Investigate how other variables effect aircraft damage and injury. You must choose **two** factors out of the following but are free to analyze more:

- Weather Condition
- Engine Type
- Number of Engines
- Phase of Flight
- Purpose of Flight

For each factor provide a discussion explaining your analysis with appropriate visualization / data summaries and interpreting your findings.

In [ ]:
weather_summary = summarize_groups(
    df.dropna(subset=["Weather.Condition.Clean"]),
    "Weather.Condition.Clean",
    min_n=50,
)
phase_summary = summarize_groups(
    df.dropna(subset=["Broad.Phase.Clean"]),
    "Broad.Phase.Clean",
    min_n=50,
)

display(weather_summary)
display(phase_summary)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.barplot(
    data=weather_summary.reset_index(),
    x="Weather.Condition.Clean",
    y="mean_injury",
    ax=axes[0, 0],
    color="tomato",
)
axes[0, 0].set_title("Weather Condition vs Fatal/Serious Injury Fraction")
axes[0, 0].set_xlabel("")
axes[0, 0].set_ylabel("Mean fatal/serious injury fraction")

sns.barplot(
    data=weather_summary.reset_index(),
    x="Weather.Condition.Clean",
    y="mean_destroyed",
    ax=axes[0, 1],
    color="lightsalmon",
)
axes[0, 1].set_title("Weather Condition vs Destroyed Fraction")
axes[0, 1].set_xlabel("")
axes[0, 1].set_ylabel("Destroyed fraction")

phase_plot_data = phase_summary.sort_values("mean_injury", ascending=False).reset_index()

sns.barplot(
    data=phase_plot_data,
    x="mean_injury",
    y="Broad.Phase.Clean",
    ax=axes[1, 0],
    color="firebrick",
)
axes[1, 0].set_title("Flight Phase vs Fatal/Serious Injury Fraction")
axes[1, 0].set_xlabel("Mean fatal/serious injury fraction")
axes[1, 0].set_ylabel("")

sns.barplot(
    data=phase_plot_data,
    x="mean_destroyed",
    y="Broad.Phase.Clean",
    ax=axes[1, 1],
    color="indianred",
)
axes[1, 1].set_title("Flight Phase vs Destroyed Fraction")
axes[1, 1].set_xlabel("Destroyed fraction")
axes[1, 1].set_ylabel("")

plt.tight_layout()

print(
    f"Weather finding: IMC accidents average a fatal/serious injury fraction of "
    f"{weather_summary.loc['IMC', 'mean_injury']:.3f}, versus "
    f"{weather_summary.loc['VMC', 'mean_injury']:.3f} in VMC."
)
print(
    f"Phase-of-flight finding: {phase_summary['mean_injury'].idxmax()} has the highest average injury fraction "
    f"({phase_summary['mean_injury'].max():.3f}), while {phase_summary['mean_destroyed'].idxmax()} has the highest destroyed fraction "
    f"({phase_summary['mean_destroyed'].max():.3f})."
)
